In [1]:
import pandas as pd
import re
import numpy as np

In [2]:
completions_raw = pd.read_csv(
    r"..\data\raw\c2024_a.csv",
    dtype=str, 
    low_memory=False
)

print(completions_raw.shape)
completions_raw.head()

(307707, 64)


,UNITID,CIPCODE,MAJORNUM,AWLEVEL,XCTOTALT,CTOTALT,XCTOTALM,CTOTALM,XCTOTALW,CTOTALW,...,XCUNKNM,CUNKNM,XCUNKNW,CUNKNW,XCNRALT,CNRALT,XCNRALM,CNRALM,XCNRALW,CNRALW
0,100654,01.0999,1,5,R,12,R,2,R,10,...,Z,0,Z,0,Z,0,Z,0,Z,0
1,100654,01.1001,1,5,R,10,R,1,R,9,...,Z,0,Z,0,Z,0,Z,0,Z,0
2,100654,01.1001,1,7,R,7,R,1,R,6,...,Z,0,Z,0,R,2,R,1,R,1
3,100654,01.1001,1,17,R,6,R,3,R,3,...,R,0,R,2,R,3,R,2,R,1
4,100654,01.9999,1,5,R,2,R,1,R,1,...,Z,0,Z,0,Z,0,Z,0,Z,0


In [3]:
# Define the columns
cols = [
    "UNITID",
    "CIPCODE",        # cip6 format (xx.xxxx)
    "AWLEVEL",        # credential level
    "CTOTALT",        # total completions
    "CTOTALM",        # male
    "CTOTALW"        # female
]

completions = completions_raw[cols].copy()

Clean columns

In [4]:
# Unitid
completions["unitid_n"] = (
    completions["UNITID"]
    .astype(str)
    .str.strip()
)

In [5]:
# cip6 to cip4
completions["cip6"] = completions["CIPCODE"].astype(str).str.strip()
completions["cip6_n"] = completions["cip6"].str.replace(".", "", regex=False)
completions["cip4"] = completions["cip6_n"].str[:4]

In [6]:
# Drop cip 99 (a placeholder for programs that cannot be traditionally coded)
completions = completions[completions["cip4"] != "99"]

In [7]:
# AWLEVEL to credlev_n
completions['credlev_n'] = pd.to_numeric(completions["AWLEVEL"], errors="coerce").astype("Int64")

In [8]:
# Completions counts
for c in ["CTOTALT", "CTOTALM", "CTOTALW"]:
    completions[c] = pd.to_numeric(completions[c], errors="coerce").fillna(0).astype(int)

completions = completions.rename(columns={
    "CTOTALT": "completions",
    "CTOTALM": "completions_men",
    "CTOTALW": "completions_women"
})

In [9]:
# Add a year column
completions["year"] = 2024

In [10]:
# Create a dictionary for the credlev codes, limiting it to the ones I will be using
credlev_map = {
    1: [2, 3, 5],  # IPEDS codes for certificates
    2: [4],     # IPEDS codes for associates
    3: [6]         # IPEDS codes for bachelors
}

def map_credlevel(awlevel):
    for proj_level, ipeds_codes in credlev_map.items():
        if awlevel in ipeds_codes:
            return proj_level
    return np.nan

completions["credlev_proj"] = completions["credlev_n"].apply(map_credlevel)

In [11]:
# Aggregate: unitid × cip4 × credlev × year
completions_2024_agg = (
    completions
    .groupby(["unitid_n", "cip4", "credlev_proj", "year"], as_index=False)
    .agg({
        "completions": "sum",
        "completions_men": "sum",
        "completions_women": "sum"
    })
)

In [12]:
# Drop rows with 0 completions
completions_2024_agg = completions_2024_agg[
    (completions_2024_agg["completions"] > 0) |
    (completions_2024_agg["completions_men"] > 0) |
    (completions_2024_agg["completions_women"] > 0)
]

In [13]:
completions_2024_agg["credlev_proj"] = completions_2024_agg["credlev_proj"].astype(int)

In [14]:
# Name credlev_proj descriptions
cred_desc_map = {
    1: "Certificate",
    2: "Assoicate",
    3: "Bachelor"
}

completions_2024_agg["cred_desc"] = completions_2024_agg["credlev_proj"].map(cred_desc_map)

In [15]:
# Create composite key
completions_2024_agg["composite_key"] = (
    completions_2024_agg["unitid_n"] + "|" +
    completions_2024_agg["cip4"] + "|" +
    completions_2024_agg["credlev_proj"].astype(str)
)

In [16]:
# uniqueness
completions_2024_agg["composite_key"].is_unique

True

In [17]:
# null checks
completions_2024_agg.isna().sum()

unitid_n             0
cip4                 0
credlev_proj         0
year                 0
completions          0
completions_men      0
completions_women    0
cred_desc            0
composite_key        0
dtype: int64

In [18]:
# sanity check
completions_2024_agg.sample(10)

,unitid_n,cip4,credlev_proj,year,completions,completions_men,completions_women,cred_desc,composite_key
111037,390905,5108,1,2024,6,2,4,Certificate,390905|5108|1
4846,108807,1002,1,2024,4,1,3,Certificate,108807|1002|1
115975,479062,4603,1,2024,138,130,8,Certificate,479062|4603|1
107528,241739,5203,1,2024,37,16,21,Certificate,241739|5203|1
63719,192749,5138,1,2024,63,8,55,Certificate,192749|5138|1
35181,155681,1601,1,2024,9,1,8,Certificate,155681|1601|1
60505,188058,4103,1,2024,1,0,1,Certificate,188058|4103|1
89035,219347,2612,1,2024,1,0,1,Certificate,219347|2612|1
20746,136215,5100,3,2024,2,0,2,Bachelor,136215|5100|3
97838,230603,1905,1,2024,29,11,18,Certificate,230603|1905|1


In [19]:
# Export
completions_2024_agg.to_csv(
    "completions_2024_model.csv",
    index=False,
    encoding="utf-8"
)

In [21]:
scorecard_tn_export = pd.read_csv("..\data\scorecard_tn_export.csv")

In [23]:
for df, cred_col in [(scorecard_tn_export, "credlev_n"), (completions, "credlev_proj")]:
    df["unitid_n"] = df["unitid_n"].astype(str).str.strip()
    df["cip4"] = df["cip4"].astype(str).str.strip()  
    df[cred_col] = df[cred_col].astype(pd.Int64Dtype()).astype(str).str.strip()

for df, cred_col in [(scorecard_tn_export, "credlev_n"), (completions, "credlev_proj")]:  
    df.dropna(subset=["unitid_n", "cip4", cred_col], inplace=True)

In [26]:
# Convert integer columns to strings before concatenation
scorecard_tn_export["composite_key"] = scorecard_tn_export["unitid_n"].astype(str) + "|" + scorecard_tn_export["cip4"].astype(str) + "|" + scorecard_tn_export["credlev_n"].astype(str)

completions_2024_agg["composite_key"] = completions_2024_agg["unitid_n"].astype(str) + "|" + completions_2024_agg["cip4"].astype(str) + "|" + completions_2024_agg["credlev_proj"].astype(str)

In [27]:
# Keep display columns from scorecard
scorecard_dim = scorecard_tn_export[[
    "composite_key","unitid_n","inst_name","cip4","cip_desc",
    "credlev_n","cred_desc","state","control"
]].rename(columns={"credlev_n":"credlev_proj"})

# For completions, we only need keys for missing rows (fill display columns with blanks)
completions_dim = completions_2024_agg[["composite_key","unitid_n","cip4","credlev_proj"]].copy()
completions_dim["inst_name"] = ""
completions_dim["cip_desc"] = ""
completions_dim["cred_desc"] = ""
completions_dim["state"] = ""
completions_dim["control"] = ""

# Combine and drop duplicates
dim_all = pd.concat([scorecard_dim, completions_dim], ignore_index=True)
dim_all = dim_all.drop_duplicates(subset=["composite_key"])

In [28]:
print("Number of unique composite keys:", dim_all["composite_key"].nunique())
print("Check for empty keys:", dim_all["composite_key"].isna().sum())

Number of unique composite keys: 99769
Check for empty keys: 0


In [30]:
dim_all.to_csv("dim_all_2024.csv", index=False, encoding="utf-8")
scorecard_tn_export.to_csv("scorecard_fact_2024.csv", index=False, encoding="utf-8")
completions_2024_agg.to_csv("completions_fact_2024.csv", index=False, encoding="utf-8")

# Scorecard composite key
scorecard_clean["composite_key"] = (
    scorecard_clean["unitid_n"] + "|" +
    scorecard_clean["cip4"] + "|" +
    scorecard_clean["credlev_n"]
)

# Completions composite key
completions_clean["composite_key"] = (
    completions_clean["unitid_n"] + "|" +
    completions_clean["cip4"] + "|" +
    completions_clean["credlev_proj"]
)

# Select columns for dimension (add display columns if needed)
scorecard_dim = scorecard_clean[["unitid_n","cip4","credlev_n","inst_name","cip_desc","cred_desc","state","control","composite_key"]]
completions_dim = completions_clean[["unitid_n","cip4","credlev_proj"]].assign(
    inst_name="", cip_desc="", cred_desc="", state="", control=""
)

# Combine and drop duplicates
dim_all = pd.concat([scorecard_dim, completions_dim], ignore_index=True)
dim_all = dim_all.drop_duplicates(subset=["composite_key"])


# Keys in both facts
set(scorecard_clean["composite_key"]) & set(completions_clean["composite_key"])

dim_all.to_csv("dim_all.csv", index=False, encoding="utf-8")
scorecard_clean.to_csv("scorecard_fact.csv", index=False, encoding="utf-8")
completions_clean.to_csv("completions_fact.csv", index=False, encoding="utf-8")